# mart_user_weekly_analysis 생성

Staging View 없이 원본 테이블을 결합해 유저 1명 × 주차 1행의 분석 Mart를 생성합니다.

## 데이터 마트 설계 목적

본 마트는 기준 주차의 유저 행동 및 관계 특성과 다음 주 재방문의 관계를 분석하기 위해 생성했다.

- 분석 단위: 유저 1명 × 기준 주차 1행
- 기준 행동: 해당 주차의 Hackle 이용 행동 및 내부 투표·친구 관계
- 결과 지표: 다음 주 1회 이상 활동 여부
- 유지 분석 대상: 다음 주 전체 기간을 관찰할 수 있는 행
- 관계 분석 대상: 내부 회원 정보와 연결 가능한 유저

행동이 발생하지 않은 횟수형 컬럼은 0으로 처리한다.  
반면 관찰 기간 부족이나 계산 대상이 아닌 경우는 0과 구분하기 위해 NULL로 유지한다.

In [15]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATASET_ID = "sns_analysis"
MAX_BYTES = 1100 * 1024**2

client = bigquery.Client(project=PROJECT_ID, location="asia-northeast3")

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## Mart 생성 쿼리

## 마트 생성 흐름

1. Hackle 로그에서 유저별 주간 활동을 집계한다.
2. 유저·주차 조합을 기준으로 행동 횟수와 활동일수를 계산한다.
3. 내부 회원 정보를 연결한다.
4. 수락된 친구 요청을 양방향 관계로 변환해 유저별 친구 수를 계산한다.
5. 투표 송수신량, 교류 상대 수, 받은 투표 집중도를 계산한다.
6. 같은 학급에서 실제 활동한 유저 수를 계산한다.
7. 기준 주차의 행동 정보에 다음 주 활동 여부를 연결한다.
8. 다음 주 전체를 관찰할 수 있는지 별도 플래그로 표시한다.

In [ ]:
mart_select_sql = f"""
-- 분석 단위: 유저 1명 × 기준 주차 1행
-- 분석 목적: 기준 주차의 이용·관계 행동과 다음 주 재방문의 관계 확인

WITH bidirectional_accepted_friend_edges AS (
    -- 친구 요청 원본은 발신자 → 수신자 한 방향으로 저장된다.
    -- 수락된 관계는 양쪽 유저의 친구 수에 포함되어야 하므로
    -- A → B를 A → B와 B → A의 양방향 관계로 확장한다.
    SELECT
        send_user_id AS user_id,
        receive_user_id AS friend_id
    FROM `{PROJECT_ID}.{DATASET_ID}.accounts_friendrequest`
    WHERE status = 'A'

    UNION DISTINCT

    SELECT
        receive_user_id AS user_id,
        send_user_id AS friend_id
    FROM `{PROJECT_ID}.{DATASET_ID}.accounts_friendrequest`
    WHERE status = 'A'
),

-- 양방향으로 확장한 관계를 기준으로 유저별 고유 친구 수 계산
user_friend_counts AS (
    SELECT
        user_id,
        COUNT(DISTINCT friend_id) AS friend_count
    FROM bidirectional_accepted_friend_edges
    GROUP BY user_id
),

-- 숫자형 내부 회원 ID가 하나로 확정되는 세션만 사용
-- 하나의 세션이 여러 회원 ID와 연결되면 유저를 확정할 수 없어 제외
session_user AS (
    SELECT
        hp.session_id,
        ANY_VALUE(SAFE_CAST(hp.user_id AS INT64)) AS user_id
    FROM `{PROJECT_ID}.{DATASET_ID}.hackle_properties` AS hp
    WHERE hp.session_id IS NOT NULL
      AND TRIM(hp.session_id) != ''
      AND SAFE_CAST(hp.user_id AS INT64) IS NOT NULL
    GROUP BY hp.session_id
    HAVING COUNT(DISTINCT hp.user_id) = 1
),

-- Hackle 이벤트를 유저×주차 단위로 집계
weekly_activity AS (
    SELECT
        s.user_id,
        DATE_TRUNC(
            DATE(e.event_datetime, 'Asia/Seoul'),
            WEEK(MONDAY)
        ) AS week_start,
        COUNT(DISTINCT DATE(e.event_datetime, 'Asia/Seoul')) AS active_days,
        COUNT(*) AS event_count,
        COUNT(DISTINCT e.session_id) AS session_count,
        COUNTIF(e.event_key = 'click_question_start') AS question_start_count,
        COUNTIF(e.event_key = 'complete_question') AS question_complete_count,
        COUNTIF(e.event_key = 'skip_question') AS question_skip_count,
        COUNTIF(e.event_key = 'click_question_open') AS received_vote_open_count,
        COUNTIF(e.event_key = 'view_timeline_tap') AS timeline_view_count,
        COUNTIF(e.event_key = 'click_attendance') AS attendance_click_count,
        COUNTIF(e.event_key = 'complete_purchase') AS purchase_count
    FROM `{PROJECT_ID}.{DATASET_ID}.hackle_events` AS e
    JOIN session_user AS s USING (session_id)
    WHERE DATE(e.event_datetime, 'Asia/Seoul')
          BETWEEN '2023-07-24' AND '2023-08-10'
    GROUP BY s.user_id, week_start
),

-- 실제 투표 데이터를 주차·투표자·수신자 조합으로 집계
vote_pairs AS (
    SELECT
        DATE_TRUNC(
            DATE(created_at, 'Asia/Seoul'),
            WEEK(MONDAY)
        ) AS week_start,
        user_id AS voter_user_id,
        chosen_user_id AS receiver_user_id,
        COUNT(*) AS pair_votes
    FROM `{PROJECT_ID}.{DATASET_ID}.accounts_userquestionrecord`
    WHERE DATE(created_at, 'Asia/Seoul')
          BETWEEN '2023-07-24' AND '2023-08-10'
    GROUP BY week_start, voter_user_id, receiver_user_id
),

-- 유저별 보낸 투표 수와 투표한 상대 수 계산
votes_given AS (
    SELECT
        week_start,
        voter_user_id AS user_id,
        SUM(pair_votes) AS votes_given,
        COUNT(*) AS unique_receivers
    FROM vote_pairs
    GROUP BY week_start, user_id
),

-- 유저별 받은 투표 수, 투표해준 상대 수, 최다 투표자의 투표 수 계산
votes_received AS (
    SELECT
        week_start,
        receiver_user_id AS user_id,
        SUM(pair_votes) AS votes_received,
        COUNT(*) AS unique_voters,
        MAX(pair_votes) AS top_voter_votes
    FROM vote_pairs
    GROUP BY week_start, user_id
)

SELECT
    -- 유저 및 소속 정보
    w.user_id,
    w.week_start,
    DATE(u.created_at, 'Asia/Seoul') AS signup_date,
    u.gender,
    u.group_id,
    gp.school_id,
    gp.grade,
    gp.class_num,
    sc.school_type,

    -- 친구 관계
    -- 내부 유저와 연결됐지만 친구 관계가 없으면 0명으로 처리
    COALESCE(f.friend_count, 0) AS friend_count,

    -- 기준 주차의 Hackle 이용 행동
    w.active_days,
    w.event_count,
    w.session_count,
    w.question_start_count,
    w.question_complete_count,
    w.question_skip_count,
    w.received_vote_open_count,
    w.timeline_view_count,
    w.attendance_click_count,
    w.purchase_count,

    -- 기준 주차의 실제 투표 관계
    -- 집계된 투표 행동이 없으면 실제 행동 횟수 0을 의미하므로 0으로 처리
    COALESCE(g.votes_given, 0) AS votes_given,
    COALESCE(g.unique_receivers, 0) AS unique_receivers,
    COALESCE(r.votes_received, 0) AS votes_received,
    COALESCE(r.unique_voters, 0) AS unique_voters,
    COALESCE(r.top_voter_votes, 0) AS top_voter_votes,

    -- 최다 투표자의 투표 수 ÷ 전체 받은 투표 수
    -- 받은 투표가 없으면 집중도를 계산할 수 없으므로 NULL 유지
    SAFE_DIVIDE(
        r.top_voter_votes,
        r.votes_received
    ) AS received_concentration,

    -- 같은 주차·학급에서 활동한 다른 유저 수
    -- 본인을 제외하기 위해 1을 차감
    CASE
        WHEN u.group_id IS NULL THEN 0
        ELSE COUNT(*) OVER (
            PARTITION BY w.week_start, u.group_id
        ) - 1
    END AS active_classmates,

    -- 기준 주차와 다음 주 전체를 모두 관찰할 수 있는지 표시
    IF(
        DATE_ADD(w.week_start, INTERVAL 13 DAY) <= DATE '2023-08-10',
        1,
        0
    ) AS is_retention_eligible,

    -- 관찰 가능한 경우에만 다음 주 활동 여부를 0 또는 1로 부여
    -- 관찰 기간이 부족하면 이탈로 오분류하지 않도록 NULL 처리
    CASE
        WHEN DATE_ADD(w.week_start, INTERVAL 13 DAY) > DATE '2023-08-10'
            THEN NULL
        WHEN n.user_id IS NOT NULL
            THEN 1
        ELSE 0
    END AS next_week_retained

FROM weekly_activity AS w

-- 동일 유저의 다음 주 활동 행을 연결해 재방문 여부 확인
LEFT JOIN weekly_activity AS n
    ON w.user_id = n.user_id
   AND n.week_start = DATE_ADD(w.week_start, INTERVAL 7 DAY)

-- Hackle 유저를 내부 회원 및 소속 정보와 연결
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.accounts_user` AS u
    ON w.user_id = u.id
LEFT JOIN user_friend_counts AS f
    ON w.user_id = f.user_id
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.accounts_group` AS gp
    ON u.group_id = gp.id
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.accounts_school` AS sc
    ON gp.school_id = sc.id

-- 기준 주차가 같은 투표 데이터만 연결
LEFT JOIN votes_given AS g
    ON w.user_id = g.user_id
   AND w.week_start = g.week_start
LEFT JOIN votes_received AS r
    ON w.user_id = r.user_id
   AND w.week_start = r.week_start
"""

## 예상 처리량 확인

In [16]:
dry_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
estimated = client.query(mart_select_sql, job_config=dry_config).total_bytes_processed

print(f"예상 처리량: {estimated / 1024**2:,.2f} MiB")
if estimated > MAX_BYTES:
    raise ValueError("1 GiB 상한을 초과했습니다.")

예상 처리량: 1,042.17 MiB


## Mart 생성

In [17]:
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.mart_user_weekly_analysis`
PARTITION BY week_start
CLUSTER BY user_id, group_id AS
{mart_select_sql}
"""

config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
client.query(sql, job_config=config).result()
print("mart_user_weekly_analysis 생성 완료")

mart_user_weekly_analysis 생성 완료


## 최종 검증

In [20]:
validation_sql = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users,
    COUNTIF(is_retention_eligible = 1) AS eligible_rows,
    COUNTIF(next_week_retained = 1) AS retained_rows
FROM `{PROJECT_ID}.{DATASET_ID}.mart_user_weekly_analysis`
"""

config = bigquery.QueryJobConfig(
    maximum_bytes_billed=MAX_BYTES
)

validation_df = client.query(
    validation_sql,
    job_config=config
).to_dataframe(create_bqstorage_client=False)

validation_df

,total_rows,unique_users,eligible_rows,retained_rows
0,273261,188235,117741,44945
